# 05 — Generous Tipper Classification

This notebook predicts whether a rider is likely to leave a generous tip (≥20%).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [ ]:
df = pd.read_csv("../data/2017_Yellow_Taxi_Trip_Data.csv")
means = pd.read_csv("../data/nyc_preds_means.csv")

# card only because cash tips are not fully observable
df = df[df["payment_type"] == 1].copy()

# target
base_amount = df["total_amount"] - df["tip_amount"]
df["tip_percent"] = np.where(base_amount > 0, df["tip_amount"] / base_amount, 0)
df["tip_percent"] = df["tip_percent"].round(3)
df["generous"] = (df["tip_percent"] >= 0.20).astype(int)

df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])
df["duration"] = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60
df["duration"] = df["duration"].clip(lower=0)
df["pickup_dropoff"] = df["PULocationID"].astype(str) + "_" + df["DOLocationID"].astype(str)
df["rush_hour"] = df["tpep_pickup_datetime"].dt.hour.isin([7,8,9,16,17,18]).astype(int)

if set(["pickup_dropoff", "mean_distance", "mean_duration"]).issubset(means.columns):
    df = df.merge(means[["pickup_dropoff", "mean_distance", "mean_duration"]], on="pickup_dropoff", how="left")
else:
    df["mean_distance"] = np.nan
    df["mean_duration"] = np.nan

feature_cols = [
    "VendorID","passenger_count","RatecodeID","trip_distance","fare_amount",
    "extra","mta_tax","tolls_amount","improvement_surcharge","total_amount",
    "duration","mean_distance","mean_duration","rush_hour","PULocationID","DOLocationID"
]

model_df = df[feature_cols + ["generous"]].copy()
model_df = pd.get_dummies(model_df, columns=["VendorID","RatecodeID","PULocationID","DOLocationID"], drop_first=True)
model_df = model_df.fillna(model_df.mean(numeric_only=True))

X = model_df.drop(columns="generous")
y = model_df["generous"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.shape, X_test.shape, y_train.mean()

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

rf_metrics = {
    "accuracy": accuracy_score(y_test, rf_preds),
    "precision": precision_score(y_test, rf_preds),
    "recall": recall_score(y_test, rf_preds),
    "f1": f1_score(y_test, rf_preds),
    "confusion_matrix": confusion_matrix(y_test, rf_preds)
}
rf_metrics

In [ ]:
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="logloss",
    random_state=42
)
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test)

xgb_metrics = {
    "accuracy": accuracy_score(y_test, xgb_preds),
    "precision": precision_score(y_test, xgb_preds),
    "recall": recall_score(y_test, xgb_preds),
    "f1": f1_score(y_test, xgb_preds),
    "confusion_matrix": confusion_matrix(y_test, xgb_preds)
}
xgb_metrics

## Validated project result

Final modeling approach:
- card transactions only
- final dataset approximately **15,265 rows × 348 columns**
- nearly balanced target (**~52.6% generous**)

**Random Forest (test)**
- Accuracy: **~0.717**
- Precision: **~0.696**
- Recall: **~0.820**
- F1: **~0.753**

**XGBoost (test)**
- F1: **~0.752**

Important variables:
- `VendorID`
- `passenger_count`
- `predicted_fare`
- `mean_distance`
- `mean_duration`

Business interpretation: if false positives are more costly because they create unrealistic expectations for drivers, the classification threshold should be tuned toward higher precision.
